# 02 · Scorer and encoder baselines (E1, E3)
1. Load the thesis per-comment prediction files (exp1–exp8, seeds 42–44, val + test).
2. Check every file is row-aligned with the gold data (ids, gold sentiment, gold sarcasm, post).
3. Convert to the §5.2 format → `data/baselines/<model>_seed<s>_<split>.csv`.
4. Score all of them with the **one shared scorer** (`src/sarcllm/eval`) and check it reproduces ICARC-MS.
5. Bootstrap sanity check (post_id primary, post_group sensitivity) and detector routing statistics.

Encoder test scores are already published (ICARC-MS), so scoring them here is not a new look at test.

In [2]:
print("Looking in:", THESIS)
print("Folder exists:", THESIS.exists())
if THESIS.exists():
    names = sorted(p.name for p in THESIS.iterdir())
    print(len(names), "files, e.g.:", names[:5])

Looking in: e:\sinhala-sarcasm-llm\data\baselines\thesis_raw
Folder exists: True
72 files, e.g.: ['exp1_baseline_s42_history.csv', 'exp1_baseline_s42_results.json', 'exp1_baseline_s42_test_preds.csv', 'exp1_baseline_s42_val_preds.csv', 'exp1_baseline_s43_history.csv']


In [3]:
import sys, re, json, datetime
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))           # use the shared scorer from src/

from sarcllm.eval.metrics import score, align, compute
from sarcllm.eval.bootstrap import ci_single, ci_paired
from sarcllm.eval.significance import mcnemar, sentiment_correct

PROC_DIR  = ROOT / "data" / "processed"
THESIS    = ROOT / "data" / "baselines" / "thesis_raw"
BASE_DIR  = ROOT / "data" / "baselines"
N_BOOT    = 10_000
pd.set_option("display.width", 200, "display.max_columns", 60)
print("thesis files:", len(list(THESIS.glob("*_preds.csv"))), "prediction CSVs")

thesis files: 36 prediction CSVs


In [4]:
def load_gold(split):
    df = pd.read_json(PROC_DIR / f"{split}.jsonl", lines=True, dtype=False)
    for c in ("id", "post_id", "post_group"):
        df[c] = df[c].astype(str)
    return df

gold = {s: load_gold(s) for s in ("val", "test")}
for s, g in gold.items():
    print(f"{s:5s} {len(g):,} rows · {g.post_id.nunique()} posts · {g.post_group.nunique()} post_groups · "
          f"{g.is_sarcastic.sum():,} sarcastic")

val   6,938 rows · 101 posts · 98 post_groups · 2,264 sarcastic
test  6,875 rows · 99 posts · 97 post_groups · 2,277 sarcastic


In [5]:
MODEL_NAMES = {
    "exp1": "xlmr_base_sent",  "exp2": "xlmr_base_mt",  "exp5": "xlmr_base_hier",
    "exp6": "xlmr_large_sent", "exp7": "xlmr_large_mt", "exp8": "xlmr_large_hier",
}
pat = re.compile(r"^(exp\d+)_.*_s(\d+)_(test|val)_preds\.csv$")
files = []
for p in sorted(THESIS.glob("*_preds.csv")):
    m = pat.match(p.name)
    if not m:
        print("skipped (name not recognised):", p.name); continue
    exp, seed, split = m.groups()
    if exp not in MODEL_NAMES:
        print("skipped (exp not in MODEL_NAMES):", p.name); continue
    files.append({"exp": exp, "model": MODEL_NAMES[exp], "seed": int(seed), "split": split, "path": p})
inv = pd.DataFrame(files)
display(inv.pivot_table(index="model", columns=["split", "seed"], values="exp", aggfunc="count", fill_value=0))

split           test       val      
seed              42 43 44  42 43 44
model                               
xlmr_base_hier     1  1  1   1  1  1
xlmr_base_mt       1  1  1   1  1  1
xlmr_base_sent     1  1  1   1  1  1
xlmr_large_hier    1  1  1   1  1  1
xlmr_large_mt      1  1  1   1  1  1
xlmr_large_sent    1  1  1   1  1  1

In [7]:
SENT_ID2LAB = {0: "negative", 1: "neutral", 2: "positive"}
preds, problems, colsets = {}, [], {}

for f in files:
    df = pd.read_csv(f["path"], dtype={"id": str, "post_id": str})
    g = gold[f["split"]].set_index("id")
    key = (f["model"], f["seed"], f["split"])
    colsets.setdefault(f["model"], set()).add(tuple(df.columns))

    required = {"id", "y_true", "y_pred", "sarc_true", "sarc_pred"}
    if not required <= set(df.columns):
        problems.append(f"{key}: missing columns {sorted(required - set(df.columns))}"); continue
    if set(df.id) != set(g.index) or df.id.duplicated().any():
        problems.append(f"{key}: id set differs from gold ({len(set(df.id) ^ set(g.index))} ids differ)"); continue

    gg = g.loc[df.id]
    checks = {
        "y_true":    (df.y_true.map(SENT_ID2LAB).values == gg.sentiment.values).mean(),
        "sarc_true": (df.sarc_true.values == gg.is_sarcastic.values).mean(),
    }
    if "post_id" in df.columns:                       # optional: not every thesis file saved it
        checks["post_id"] = (df.post_id.values == gg.post_id.values).mean()
    for k, v in checks.items():
        if v < 1:
            problems.append(f"{key}: {k} agrees on only {v:.2%} of rows")
    preds[key] = df

print("Columns per model:")
for m, cs in sorted(colsets.items()):
    for c in cs:
        print(f"  {m:16s} {list(c)}")
print()
print("\n".join(problems) or f"All {len(preds)} files aligned with gold (ids, labels{', posts where saved'}).")
assert not problems

Columns per model:
  xlmr_base_hier   ['id', 'y_true', 'y_pred', 'sarc_true', 'sarc_pred']
  xlmr_base_hier   ['id', 'post_id', 'y_true', 'y_pred', 'sarc_true', 'sarc_pred']
  xlmr_base_mt     ['id', 'y_true', 'y_pred', 'sarc_true', 'sarc_pred']
  xlmr_base_mt     ['id', 'post_id', 'y_true', 'y_pred', 'sarc_true', 'sarc_pred']
  xlmr_base_sent   ['id', 'y_true', 'y_pred', 'sarc_true', 'sarc_pred']
  xlmr_base_sent   ['id', 'post_id', 'y_true', 'y_pred', 'sarc_true', 'sarc_pred']
  xlmr_large_hier  ['id', 'y_true', 'y_pred', 'sarc_true', 'sarc_pred']
  xlmr_large_hier  ['id', 'post_id', 'y_true', 'y_pred', 'sarc_true', 'sarc_pred']
  xlmr_large_mt    ['id', 'y_true', 'y_pred', 'sarc_true', 'sarc_pred']
  xlmr_large_mt    ['id', 'post_id', 'y_true', 'y_pred', 'sarc_true', 'sarc_pred']
  xlmr_large_sent  ['id', 'y_true', 'y_pred', 'sarc_true', 'sarc_pred']
  xlmr_large_sent  ['id', 'post_id', 'y_true', 'y_pred', 'sarc_true', 'sarc_pred']

All 36 files aligned with gold (ids, labels, posts

In [8]:
def to_contract(df):
    has_sarc = (df.sarc_pred != -1).any()
    return pd.DataFrame({
        "id": df.id,
        "sentiment_pred": df.y_pred.map(SENT_ID2LAB),
        "p_neg": np.nan, "p_neu": np.nan, "p_pos": np.nan,          # not saved by the thesis
        "is_sarcastic_pred": df.sarc_pred if has_sarc else np.nan,
        "p_sarcastic": np.nan,                                        # not saved -> HY2 needs a re-run
    })

baselines = {}
for (model, seed, split), df in preds.items():
    out = to_contract(df)
    assert out.sentiment_pred.notna().all(), f"{model} s{seed} {split}: unknown y_pred codes"
    if out.is_sarcastic_pred.notna().any():
        assert set(out.is_sarcastic_pred.unique()) <= {0, 1}, f"{model} s{seed}: sarc_pred not 0/1"
    out.to_csv(BASE_DIR / f"{model}_seed{seed}_{split}.csv", index=False)
    baselines[(model, seed, split)] = out

has_det = sorted({m for (m, s, sp), d in baselines.items() if d.is_sarcastic_pred.notna().any()})
print(f"wrote {len(baselines)} files to data/baselines/   ·   models with sarcasm predictions: {has_det}")

wrote 36 files to data/baselines/   ·   models with sarcasm predictions: ['xlmr_base_hier', 'xlmr_base_mt', 'xlmr_large_hier', 'xlmr_large_mt']


In [9]:
KEY_METRICS = ["sent_f1_overall", "sent_f1_sarcastic", "sent_f1_nonsarcastic", "penalty",
               "neg_overpred_neu", "neg_overpred_pos", "det_f1_macro", "det_f1_sarcastic", "routing_recall"]

rows = []
for (model, seed, split), pr in baselines.items():
    m = score(gold[split], pr, by_script=False)
    rows.append({"model": model, "seed": seed, "split": split, **{k: m.get(k, np.nan) for k in KEY_METRICS}})
scores = pd.DataFrame(rows).sort_values(["split", "model", "seed"])

summary = scores.groupby(["split", "model"])[KEY_METRICS].agg(["mean", "std"])
display(scores[scores.split == "test"].set_index(["model", "seed"])[KEY_METRICS].round(4))
display(summary.loc["test"].round(4))

sent_f1_overall  sent_f1_sarcastic  sent_f1_nonsarcastic  penalty  neg_overpred_neu  neg_overpred_pos  det_f1_macro  det_f1_sarcastic  routing_recall
model           seed                                                                                                                                                       
xlmr_base_hier  42             0.6334             0.4325                0.6544   0.2219            0.4855            0.6864        0.7438            0.6612          0.6768
                43             0.6297             0.4270                0.6495   0.2224            0.4273            0.7203        0.7309            0.6702          0.7835
                44             0.6296             0.4331                0.6532   0.2201            0.4651            0.6949        0.7404            0.6703          0.7383
xlmr_base_mt    42             0.6263             0.4276                0.6481   0.2204            0.4709            0.6780        0.7344            0.6678          0.7554
                43             0.6167             0.4410                0.6345   0.1935            0.4913            0.6695        0.7418            0.6632          0.6965
                44             0.6308             0.4376                0.6502   0.2126            0.4448            0.6525        0.7508            0.6722          0.6948
xlmr_base_sent  42             0.6258             0.4369                0.6451   0.2083            0.4709            0.7034           NaN               NaN             NaN
                43             0.6279             0.4434                0.6429   0.1995            0.5058            0.7119           NaN               NaN             NaN
                44             0.6146             0.4360                0.6338   0.1978            0.5058            0.7034           NaN               NaN             NaN
xlmr_large_hier 42             0.6512             0.4574                0.6758   0.2185            0.4331            0.6186        0.7529            0.6791          0.7189
                43             0.6527             0.4510                0.6701   0.2190            0.4651            0.7203        0.7365            0.6839          0.8261
                44             0.6409             0.4378                0.6684   0.2306            0.4186            0.6186        0.7524            0.6800          0.7260
xlmr_large_mt   42             0.6536             0.4664                0.6727   0.2064            0.4360            0.6780        0.7578            0.6811          0.7027
                43             0.6441             0.4499                0.6660   0.2160            0.4680            0.7034        0.7510            0.6775          0.7211
                44             0.6526             0.4344                0.6785   0.2441            0.4099            0.7034        0.7554            0.6793          0.7066
xlmr_large_sent 42             0.6428             0.4517                0.6623   0.2106            0.3953            0.6441           NaN               NaN             NaN
                43             0.6483             0.4404                0.6655   0.2251            0.5145            0.7373           NaN               NaN             NaN
                44             0.6473             0.4711                0.6656   0.1945            0.4041            0.6356           NaN               NaN             NaN

sent_f1_overall         sent_f1_sarcastic         sent_f1_nonsarcastic         penalty         neg_overpred_neu         neg_overpred_pos         det_f1_macro          \
                           mean     std              mean     std                 mean     std    mean     std             mean     std             mean     std         mean     std   
model                                                                                                                                                                                   
xlmr_base_hier           0.6309  0.0022            0.4309  0.0033               0.6523  0.0025  0.2215  0.0012           0.4593  0.0295           0.7006  0.0176       0.7384  0.0067   
xlmr_base_mt             0.6246  0.0072            0.4354  0.0069               0.6443  0.0085  0.2088  0.0139           0.4690  0.0233           0.6667  0.0129       0.7423  0.0082   
xlmr_base_sent           0.6228  0.0071            0.4388  0.0041               0.6406  0.0060  0.2019  0.0056           0.4942  0.0201           0.7062  0.0049          NaN     NaN   
xlmr_large_hier          0.6483  0.0064            0.4487  0.0100               0.6714  0.0039  0.2227  0.0068           0.4390  0.0238           0.6525  0.0587       0.7473  0.0093   
xlmr_large_mt            0.6501  0.0052            0.4502  0.0160               0.6724  0.0063  0.2222  0.0196           0.4380  0.0291           0.6949  0.0147       0.7547  0.0035   
xlmr_large_sent          0.6461  0.0030            0.4544  0.0155               0.6645  0.0019  0.2101  0.0153           0.4380  0.0664           0.6723  0.0564          NaN     NaN   

                det_f1_sarcastic         routing_recall          
                            mean     std           mean     std  
model                                                            
xlmr_base_hier            0.6672  0.0052         0.7328  0.0536  
xlmr_base_mt              0.6677  0.0045         0.7156  0.0345  
xlmr_base_sent               NaN     NaN            NaN     NaN  
xlmr_large_hier           0.6810  0.0025         0.7570  0.0599  
xlmr_large_mt             0.6793  0.0018         0.7101  0.0097  
xlmr_large_sent              NaN     NaN            NaN     NaN

In [10]:
EXPECTED = {
    ("xlmr_large_sent", "sent_f1_overall"):   0.6461,
    ("xlmr_large_mt",   "sent_f1_overall"):   0.6501,
    ("xlmr_large_sent", "sent_f1_sarcastic"): 0.4544,
    ("xlmr_large_mt",   "sent_f1_sarcastic"): 0.4502,
    ("xlmr_large_mt",   "det_f1_macro"):      0.7547,
}
TOL = 5e-4
test_means = scores[scores.split == "test"].groupby("model")[KEY_METRICS].mean()
rep = pd.DataFrame([{"model": m, "metric": k, "ICARC-MS": v, "this scorer": test_means.loc[m, k]}
                    for (m, k), v in EXPECTED.items()])
rep["diff"] = rep["this scorer"] - rep["ICARC-MS"]
rep["ok"] = rep["diff"].abs() <= TOL
display(rep.round(4))
if rep.ok.all():
    print("Scorer reproduces ICARC-MS: PASS")
else:
    print("MISMATCH — compare with the thesis results.json below before going further.")

,model,metric,ICARC-MS,this scorer,diff,ok
0,xlmr_large_sent,sent_f1_overall,0.6461,0.6461,0.0,True
1,xlmr_large_mt,sent_f1_overall,0.6501,0.6501,0.0,True
2,xlmr_large_sent,sent_f1_sarcastic,0.4544,0.4544,0.0,True
3,xlmr_large_mt,sent_f1_sarcastic,0.4502,0.4502,0.0,True
4,xlmr_large_mt,det_f1_macro,0.7547,0.7547,0.0,True


Scorer reproduces ICARC-MS: PASS


In [11]:
def flat(d, pre=""):
    out = {}
    for k, v in d.items():
        if isinstance(v, dict):
            out.update(flat(v, f"{pre}{k}."))
        elif isinstance(v, (int, float)) and not isinstance(v, bool):
            out[f"{pre}{k}"] = v
    return out

js = {}
for p in sorted(THESIS.glob("exp[678]_*_results.json")):
    m = re.match(r"^(exp\d+)_.*_s(\d+)_results\.json$", p.name)
    js[(MODEL_NAMES[m.group(1)], int(m.group(2)))] = flat(json.loads(p.read_text(encoding="utf-8")))
thesis_json = pd.DataFrame(js).T
display(thesis_json.round(4))

seed  epochs_run  best_val_sent_f1_during_training  test_tokenization_val_match  val.sent_macro_f1  val.sent_acc  val.sent_f1_per_class.Negative  val.sent_f1_per_class.Neutral  \
xlmr_large_sent 42  42.0         7.0                            0.6277                          1.0             0.6276        0.6650                          0.7543                         0.5598   
                43  43.0         7.0                            0.6327                          1.0             0.6326        0.6855                          0.7812                         0.5534   
                44  44.0         9.0                            0.6403                          1.0             0.6407        0.6777                          0.7665                         0.5655   
xlmr_large_mt   42  42.0        10.0                            0.6375                          1.0             0.6379        0.6769                          0.7643                         0.5633   
                43  43.0         9.0                            0.6372                          1.0             0.6370        0.6783                          0.7670                         0.5646   
                44  44.0         6.0                            0.6292                          1.0             0.6287        0.6627                          0.7477                         0.5698   
xlmr_large_hier 42  42.0        10.0                            0.6353                          1.0             0.6352        0.6737                          0.7625                         0.5627   
                43  43.0         7.0                            0.6394                          1.0             0.6384        0.6777                          0.7668                         0.5563   
                44  44.0        10.0                            0.6378                          1.0             0.6381        0.6763                          0.7649                         0.5705   

                    val.sent_f1_per_class.Positive  val.sarc_subset_f1  val.nonsarc_subset_f1  val.penalty  test.sent_macro_f1  test.sent_acc  test.sent_f1_per_class.Negative  \
xlmr_large_sent 42                          0.5689              0.4792                 0.6473       0.1681              0.6428         0.6807                           0.7675   
                43                          0.5633              0.4715                 0.6514       0.1800              0.6483         0.6956                           0.7814   
                44                          0.5901              0.4957                 0.6599       0.1642              0.6473         0.6832                           0.7648   
xlmr_large_mt   42                          0.5861              0.4839                 0.6591       0.1752              0.6536         0.6950                           0.7796   
                43                          0.5796              0.4947                 0.6538       0.1592              0.6441         0.6851                           0.7716   
                44                          0.5686              0.4808                 0.6482       0.1675              0.6526         0.6826                           0.7563   
xlmr_large_hier 42                          0.5804              0.4780                 0.6593       0.1813              0.6512         0.6880                           0.7698   
                43                          0.5922              0.4673                 0.6613       0.1940              0.6527         0.6938                           0.7779   
                44                          0.5789              0.4923                 0.6573       0.1650              0.6409         0.6774                           0.7606   

                    test.sent_f1_per_class.Neutral  test.sent_f1_per_class.Positive  test.sarc_subset_f1  test.nonsarc_subset_f1  test.penalty  val.sarc_macro_f1  val.sarc_class_f1  val.sarc_acc  \
xlmr_large_sent 42                          0.5767

In [12]:
te = gold["test"]
ids = te.id.tolist()
units = {"post_id": te.post_id.to_numpy(), "post_group": te.post_group.to_numpy()}
CI_METRICS = ["sent_f1_overall", "sent_f1_sarcastic", "sent_f1_nonsarcastic", "penalty"]

ci_rows = []
for seed in (42, 43, 44):
    a = align(te, baselines[("xlmr_large_mt", seed, "test")], ids)
    b = align(te, baselines[("xlmr_large_sent", seed, "test")], ids)
    b.p_sarc = None
    for unit, pid in units.items():
        r = ci_paired(a, b, pid, n_boot=N_BOOT, seed=42)
        for k in CI_METRICS:
            ci_rows.append({"seed": seed, "unit": unit, "metric": k,
                            **r.loc[k, ["estimate", "ci_lo", "ci_hi", "p_boot"]].to_dict()})
ci = pd.DataFrame(ci_rows)
display(ci.pivot_table(index=["metric", "seed"], columns="unit", values=["ci_lo", "ci_hi"]).round(4))
print("Width change post_group vs post_id (mean):",
      round((ci.query("unit=='post_group'").eval("ci_hi-ci_lo").mean()
             / ci.query("unit=='post_id'").eval("ci_hi-ci_lo").mean() - 1) * 100, 1), "%")

ci_hi              ci_lo        
unit                      post_group post_id post_group post_id
metric               seed                                      
penalty              42       0.0170  0.0160    -0.0261 -0.0259
                     43       0.0166  0.0160    -0.0364 -0.0366
                     44       0.0826  0.0818     0.0187  0.0192
sent_f1_nonsarcastic 42       0.0238  0.0239    -0.0041 -0.0040
                     43       0.0129  0.0126    -0.0124 -0.0123
                     44       0.0272  0.0272    -0.0016 -0.0014
sent_f1_overall      42       0.0229  0.0227    -0.0023 -0.0020
                     43       0.0085  0.0084    -0.0174 -0.0170
                     44       0.0189  0.0190    -0.0085 -0.0084
sent_f1_sarcastic    42       0.0339  0.0330    -0.0045 -0.0030
                     43       0.0347  0.0353    -0.0144 -0.0137
                     44      -0.0084 -0.0095    -0.0644 -0.0641

Width change post_group vs post_id (mean): 1.8 %


In [13]:
mc = []
for seed in (42, 43, 44):
    a = align(te, baselines[("xlmr_large_mt", seed, "test")], ids)
    b = align(te, baselines[("xlmr_large_sent", seed, "test")], ids)
    sar = a.y_sarc == 1
    for name, mask in (("all", None), ("sarcastic", sar)):
        mc.append({"seed": seed, "subset": name, **mcnemar(sentiment_correct(a, mask), sentiment_correct(b, mask))})
display(pd.DataFrame(mc).round(4))

,seed,subset,a_only,b_only,chi2,p
0,42,all,526,428,9.8627,0.0017
1,42,sarcastic,177,136,5.1118,0.0238
2,43,all,419,491,5.5396,0.0186
3,43,sarcastic,130,191,11.2150,0.0008
4,44,all,500,504,0.0090,0.9246
5,44,sarcastic,132,187,9.1411,0.0025


In [14]:
route = []
for split in ("val", "test"):
    g = gold[split]
    flagged = {}
    for seed in (42, 43, 44):
        pr = baselines[("xlmr_large_mt", seed, split)]
        m = score(g, pr, by_script=True)
        flagged[seed] = set(pr.id[pr.is_sarcastic_pred == 1])
        route.append({"split": split, "seed": seed, "routed": len(flagged[seed]),
                      "routed_share": m["routed_share"], "routing_recall": m["routing_recall"],
                      **{f"recall_{s}": m.get(f"{s}/routing_recall") for s in ("sinhala", "latin", "mixed")}})
    union = set.union(*flagged.values()); inter = set.intersection(*flagged.values())
    route.append({"split": split, "seed": "union", "routed": len(union)})
    route.append({"split": split, "seed": "all 3", "routed": len(inter)})
display(pd.DataFrame(route).round(4))
print("'union' ≈ unique LLM calls needed for HY1 across the 3 seeds (the cache serves the rest).")

,split,seed,routed,routed_share,routing_recall,recall_sinhala,recall_latin,recall_mixed
0,val,42,2390,0.3445,0.6992,0.7487,0.5294,0.7233
1,val,43,2528,0.3644,0.7204,0.7567,0.5861,0.7533
2,val,44,2406,0.3468,0.7080,0.7540,0.5483,0.7333
3,val,union,2921,NaN,NaN,NaN,NaN,NaN
4,val,all 3,1993,NaN,NaN,NaN,NaN,NaN
5,test,42,2421,0.3521,0.7027,0.7393,0.5656,0.7198
6,test,43,2570,0.3738,0.7211,0.7701,0.5588,0.7168
7,test,44,2460,0.3578,0.7066,0.7667,0.5113,0.6962
8,test,union,2999,NaN,NaN,NaN,NaN,NaN
9,test,all 3,1975,NaN,NaN,NaN,NaN,NaN


'union' ≈ unique LLM calls needed for HY1 across the 3 seeds (the cache serves the rest).


In [15]:
tm = test_means
print(f"""## {datetime.date.today()} · Encoder baselines imported and scorer verified (notebook 02)
- Thesis prediction files: {len(preds)} (models {sorted({k[0] for k in preds})}, seeds 42–44, val+test); all aligned with gold.
- Labels only: thesis saved no probabilities → p_* and p_sarcastic empty. HY1 runs from labels; HY2 needs a multi-task re-run.
- Reproduction vs ICARC-MS: {'PASS' if rep.ok.all() else 'FAIL — see notebook 02 Cell 8'} (max |diff| {rep['diff'].abs().max():.4f}).
- Test means: large_sent overall {tm.loc['xlmr_large_sent','sent_f1_overall']:.4f} / sarc {tm.loc['xlmr_large_sent','sent_f1_sarcastic']:.4f};
  large_mt overall {tm.loc['xlmr_large_mt','sent_f1_overall']:.4f} / sarc {tm.loc['xlmr_large_mt','sent_f1_sarcastic']:.4f} / det {tm.loc['xlmr_large_mt','det_f1_macro']:.4f};
  large_hier overall {tm.loc['xlmr_large_hier','sent_f1_overall']:.4f} / sarc {tm.loc['xlmr_large_hier','sent_f1_sarcastic']:.4f}.
- [DECISION proposed] Add E3 = exp8 hierarchical large as an encoder-only detect-then-resolve comparison for HY1.
- Bootstrap: {N_BOOT} resamples, seed 42; post_group unit changes CI width by the amount printed in Cell 10.
""")

## 2026-11-21 · Encoder baselines imported and scorer verified (notebook 02)
- Thesis prediction files: 36 (models ['xlmr_base_hier', 'xlmr_base_mt', 'xlmr_base_sent', 'xlmr_large_hier', 'xlmr_large_mt', 'xlmr_large_sent'], seeds 42–44, val+test); all aligned with gold.
- Labels only: thesis saved no probabilities → p_* and p_sarcastic empty. HY1 runs from labels; HY2 needs a multi-task re-run.
- Reproduction vs ICARC-MS: PASS (max |diff| 0.0000).
- Test means: large_sent overall 0.6461 / sarc 0.4544;
  large_mt overall 0.6501 / sarc 0.4502 / det 0.7547;
  large_hier overall 0.6483 / sarc 0.4487.
- [DECISION proposed] Add E3 = exp8 hierarchical large as an encoder-only detect-then-resolve comparison for HY1.
- Bootstrap: 10000 resamples, seed 42; post_group unit changes CI width by the amount printed in Cell 10.

